# Trực quan hóa kết quả Segmentation trên các Patch (UAV / Satellite)

Mục tiêu của notebook này là trích xuất thử một vài patch từ ảnh vệ tinh siêu to (bằng `iter_patches`), đẩy qua mô hình Mask R-CNN (bằng `segment_image`) và trực quan hóa:
1. Ảnh patch gốc.
2. Binary mask (dự đoán từ model).
3. Ảnh overlay với các contour (polygons) được vẽ đè lên dưới dạng vector để kiểm tra độ chính xác của thuật toán Douglas-Peucker.

In [55]:
import sys
from pathlib import Path
REPO_ROOT = r"D:\bk_study_stuff\paper6\LocalizationUAV"
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

In [56]:
import sys
from pathlib import Path
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import os

# Import các module từ dự án của bạn (điều chỉnh đường dẫn import tùy theo cấu trúc thư mục thực tế)
# Giả sử cấu trúc package của bạn là thư mục hiện tại hoặc đã được thêm vào sys.path
# package_dir = r"D:\bk_study_stuff\paper6\LocalizationUAV" # Hoặc điền đường dẫn tuyệt đối dạng 'C:/Users/...'
# if package_dir not in sys.path:
#     sys.path.insert(0, package_dir)


from localization.segmentation.model import load_model
from localization.segmentation.inference import segment_image
from localization.database.patches import iter_patches

In [57]:
# ==========================================
# CẤU HÌNH ĐƯỜNG DẪN
# ==========================================
CHECKPOINT_PATH = r"C:\Users\Acer\Downloads\final_model.pth"  # Thay bằng đường dẫn tới file trọng số của bạn
SATELLITE_IMG_PATH = r"D:\bk_study_stuff\paper6\UAV-VisLoc\01\satellite01.tif"  # Thay bằng ảnh vệ tinh thực tế

PATCH_SIZE = 500
STRIDE = 100

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

# ==========================================
# LOAD MODEL
# ==========================================
# Lưu ý: Nếu ảnh tif của bạn có 4 kênh (RGB+NIR), hãy đổi in_channels=4
model = load_model(
    model_path=CHECKPOINT_PATH, 
    device=device, 
    num_classes=2, 
    pretrained=False, 
    in_channels=3
).to(device)
print("Load model Mask R-CNN thành công!")

Đang sử dụng thiết bị: cuda
Load model Mask R-CNN thành công!


In [58]:
# Hàm iter_patches sẽ load ảnh lười (lazy load) và cắt dần

patch_generator = iter_patches(
    image_or_path=SATELLITE_IMG_PATH, 
    patch_size=PATCH_SIZE, 
    stride=STRIDE
)

print(f"Sẵn sàng cắt patch kích thước {PATCH_SIZE}x{PATCH_SIZE}.")

Sẵn sàng cắt patch kích thước 500x500.


In [ ]:
import os
from pathlib import Path

# ==========================================================
# CẤU HÌNH THƯ MỤC LƯU VÀ KHOẢNG PATCH CẦN BÓC
# ==========================================================
OUTPUT_DIR = Path(r"D:\bk_study_stuff\paper6\UAV-VisLoc\01\output_visualizations")  # Tên thư mục bạn muốn lưu ảnh
OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # Tự động tạo thư mục nếu chưa có

START_PATCH = 501   # Bắt đầu lưu từ patch thứ mấy
END_PATCH = 700    # Dừng lại ở patch thứ mấy (Nhớ đảm bảo nhỏ hơn hoặc bằng max_patches)

# Khởi tạo Generator với biến max_patches giống như bạn muốn
patch_generator = iter_patches(
    image_or_path=SATELLITE_IMG_PATH, 
    patch_size=PATCH_SIZE, 
    stride=STRIDE, 
    max_patches=None  # Hoặc số lượng tối đa bạn cấu hình
)

print(f"Sẵn sàng xử lý! Ảnh kết quả sẽ được lưu vào thư mục: '{OUTPUT_DIR}'")
print(f"Khoảng patch cấu hình: Từ {START_PATCH} đến {END_PATCH}\n")

# Duyệt qua các patch bằng vòng lặp for kèm enumerate để lấy index
for idx, (patch_pil, patch_id, top_left_coord) in enumerate(patch_generator):
    
    # Nếu chưa đến khoảng cần lấy thì bỏ qua (skip)
    if idx < START_PATCH:
        continue
        
    # Nếu vượt quá khoảng cần lấy thì dừng vòng lặp (break)
    if idx > END_PATCH:
        break

    print(f"--> Đang xử lý [Patch Index {idx}]: {patch_id} tại tọa độ {top_left_coord}...")

    # 1. Chạy Inference (Model predict)
    binary_mask, polygons = segment_image(
        image=patch_pil,
        model=model,
        device=device,
        score_threshold=0.5,
        min_area=50.0,            
        tolerance_px=2.0,         
        contour_method="marching_squares"
    )

    # 2. Khởi tạo khung vẽ bằng Matplotlib
    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    # Panel 1: Ảnh patch gốc
    axes[0].imshow(patch_pil)
    axes[0].set_title(f"Patch Gốc\n(Tọa độ: {top_left_coord})", fontsize=14)
    axes[0].axis('off')

    # Panel 2: Binary Mask
    axes[1].imshow(binary_mask, cmap='gray')
    axes[1].set_title("Model Output (Binary Mask)", fontsize=14)
    axes[1].axis('off')

    # Panel 3: Ảnh gốc + Vẽ đè Polygon
    axes[2].imshow(patch_pil)
    axes[2].set_title(f"Polygons Overlay ({len(polygons)} buildings)", fontsize=14)
    axes[2].axis('off')

    # Vẽ từng polygon (contour) lên trục thứ 3
    for poly in polygons:
        poly_array = np.array(poly)
        
        # Vẽ mask trong suốt bên trong
        polygon_fill = mpatches.Polygon(poly_array, closed=True, alpha=0.3, facecolor='lime')
        axes[2].add_patch(polygon_fill)
        
        # Vẽ đường viền đỏ bao quanh
        polygon_edge = mpatches.Polygon(poly_array, closed=True, fill=False, edgecolor='red', linewidth=1.5)
        axes[2].add_patch(polygon_edge)

    # ==========================================================
    # TIẾN HÀNH LƯU FILE VÀO THƯ MỤC
    # ==========================================================
    # Đặt tên file kết hợp index và patch_id để không bị trùng và dễ tìm kiếm
    filename = f"visualize_idx_{idx:03d}_{patch_id}.png"
    save_path = OUTPUT_DIR / filename
    
    # Lưu ảnh xuống ổ cứng
    plt.savefig(save_path, dpi=120)
    
    # LƯU Ý CỰC KỲ QUAN TRỌNG: Phải đóng fig sau khi lưu để giải phóng RAM/VRAM. 
    # Nếu không đóng, Jupyter vẽ 50-100 cái ảnh liên tục sẽ làm crash RAM máy bạn ngay lập tức.
    plt.close(fig)

print(f"\n[DONE] Đã lưu thành công các ảnh từ patch {START_PATCH} đến {END_PATCH} vào thư mục '{OUTPUT_DIR}'!")

Sẵn sàng xử lý! Ảnh kết quả sẽ được lưu vào thư mục: 'D:\bk_study_stuff\paper6\UAV-VisLoc\01\output_visualizations'
Khoảng patch cấu hình: Từ 501 đến 700

--> Đang xử lý [Patch Index 501]: satellite01_0005_0031 tại tọa độ (3100, 500)...
--> Đang xử lý [Patch Index 502]: satellite01_0005_0032 tại tọa độ (3200, 500)...
--> Đang xử lý [Patch Index 503]: satellite01_0005_0033 tại tọa độ (3300, 500)...
--> Đang xử lý [Patch Index 504]: satellite01_0005_0034 tại tọa độ (3400, 500)...
--> Đang xử lý [Patch Index 505]: satellite01_0005_0035 tại tọa độ (3500, 500)...
--> Đang xử lý [Patch Index 506]: satellite01_0005_0036 tại tọa độ (3600, 500)...
--> Đang xử lý [Patch Index 507]: satellite01_0005_0037 tại tọa độ (3700, 500)...
--> Đang xử lý [Patch Index 508]: satellite01_0005_0038 tại tọa độ (3800, 500)...
--> Đang xử lý [Patch Index 509]: satellite01_0005_0039 tại tọa độ (3900, 500)...
--> Đang xử lý [Patch Index 510]: satellite01_0005_0040 tại tọa độ (4000, 500)...
--> Đang xử lý [Patch Ind